####DAY 6 (25/02/26) – Model Training & Tuning
####🏗️ Architecture & Strategy
Welcome to Day 6! Today is the day we actually train our AI to predict user behavior. We will build, track, and tune multiple models using Spark MLlib and MLflow .

####Our Strategy:

* **Feature Assembly (No Leakage)**: We will use VectorAssembler to pack only pre-purchase behavioral features (view_count, cart_count, total_events) into a single dense vector.

* **Baseline Model**: Train a Logistic Regression model. It's fast, interpretable, and serves as a great baseline.

* **Advanced Model & Tuning**: Train a Random Forest Classifier and use a CrossValidator to tune hyperparameters (maxDepth, numTrees).

* **MLflow Tracking**: We will wrap every training run in mlflow.start_run() to track our parameters, log our evaluation metric (AUC), and securely register the model artifacts.

####Feature Assembly & Preventing Data Leakage
Spark ML requires all input features to be combined into a single column (usually named features). We will use VectorAssembler to do this.

In [0]:
from pyspark.ml.feature import VectorAssembler

# ---------------------------------------------------------
# 1. SETUP ENVIRONMENT & SECURITY VOLUMES
# ---------------------------------------------------------
catalog_name = "course_catalog"  
schema_name = "ecommerce_governed"
volume_name = "ml_assets"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

# Create a secure UC Volume for MLflow to stage model artifacts
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")
mlflow_tmp_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/mlflow_staging"

print("⏳ Loading Gold Training and Testing Datasets...")
train_df = spark.table("gold_train_set")
test_df = spark.table("gold_test_set")

# ---------------------------------------------------------
# 2. FEATURE ASSEMBLY (No Data Leakage!)
# ---------------------------------------------------------
# We explicitly EXCLUDE post-purchase features like 'total_spent'
feature_cols = ["total_events", "view_count", "cart_count"]
print(f"🧩 Assembling Pre-Purchase Features: {feature_cols}")

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip" # Safely drops any rows with unexpected nulls
)

# Transform the datasets into Spark ML format
train_data = assembler.transform(train_df).select("user_id", "features", "purchased")
test_data = assembler.transform(test_df).select("user_id", "features", "purchased")

display(train_data.limit(5))

####Baseline Model (Logistic Regression) + MLflow
We will train a simple Logistic Regression model and log the results to MLflow. Since our data is highly imbalanced (as seen on Day 5), we will use Area Under the ROC Curve (AUC) instead of standard Accuracy .

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import mlflow

# ---------------------------------------------------------
# UNITY CATALOG VOLUME SETUP FOR MLFLOW
# ---------------------------------------------------------
catalog_name = "course_catalog"
schema_name = "ecommerce_governed"
volume_name = "ml_assets"

# Ensure a secure volume exists to stage the model files
spark.sql(f"CREATE VOLUME IF NOT EXISTS {volume_name}")
mlflow_tmp_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/mlflow_staging"

# Define the Evaluator (AUC is default for BinaryClassificationEvaluator)
evaluator = BinaryClassificationEvaluator(labelCol="purchased", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Set up MLflow Experiment
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
mlflow.set_experiment(f"/Users/{username}/Day6_Purchase_Prediction")

print("🚀 Training Baseline: Logistic Regression...")

with mlflow.start_run(run_name="Logistic_Regression_Baseline"):
    # 1. Initialize and Train Model
    lr = LogisticRegression(featuresCol="features", labelCol="purchased", maxIter=10)
    
    # Log parameters manually
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("maxIter", 10)
    
    lr_model = lr.fit(train_data)
    
    # 2. Evaluate on Test Data
    predictions = lr_model.transform(test_data)
    auc_score = evaluator.evaluate(predictions)
    
    # 3. Log Metrics and Model to MLflow
    mlflow.log_metric("test_auc", auc_score)
    mlflow.spark.log_model(
        spark_model=lr_model, 
        artifact_path="lr_model", 
        dfs_tmpdir=mlflow_tmp_path
    )
    
    print(f"   ✅ Logistic Regression AUC: {auc_score:.4f}")

####Hyperparameter Tuning (Random Forest)
Now we bring in the heavy artillery. Random Forests capture complex, non-linear relationships. We will use CrossValidator and ParamGridBuilder to test multiple configurations automatically and find the absolute best model.

In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import mlflow

print("🌲 Training & Tuning: Random Forest Classifier (Fast-Track)...")

# 1. ⚠️ THE FIX: Downsample the training data to 10% for faster tuning
# This reduces 4.2M rows to ~420k rows, which is plenty for a portfolio project
fast_train_data = train_data.sample(withReplacement=False, fraction=0.10, seed=42)

with mlflow.start_run(run_name="Random_Forest_Tuned_Fast"):
    mlflow.log_param("model_type", "RandomForest")
    
    # Initialize Base Model
    rf = RandomForestClassifier(featuresCol="features", labelCol="purchased", seed=42)
    
    # 2. ⚠️ THE FIX: Lighter Hyperparameter Grid
    paramGrid = (ParamGridBuilder()
                 .addGrid(rf.maxDepth, [5])       # Just test depth 5 for speed
                 .addGrid(rf.numTrees, [20, 50])  # Smaller forests
                 .build())
    
    # 3. ⚠️ THE FIX: 2-Fold CV instead of 3
    cv = CrossValidator(
        estimator=rf,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator, 
        numFolds=2, 
        seed=42
    )
    
    print("   ⏳ Running Grid Search on a 10% sample (Should take 1-3 minutes)...")
    cv_model = cv.fit(fast_train_data) # Feed it the sampled data!
    
    best_rf_model = cv_model.bestModel
    
    # Evaluate on the full test dataset (to see how well the sampled model generalizes)
    rf_predictions = best_rf_model.transform(test_data)
    rf_auc_score = evaluator.evaluate(rf_predictions)
    
    # Log Best Parameters, Metrics, and Model
    mlflow.log_param("best_maxDepth", best_rf_model.getOrDefault("maxDepth"))
    mlflow.log_param("best_numTrees", best_rf_model.getOrDefault("numTrees"))
    mlflow.log_metric("test_auc", rf_auc_score)
    
    mlflow.spark.log_model(
        spark_model=best_rf_model, 
        artifact_path="rf_best_model", 
        dfs_tmpdir=mlflow_tmp_path
    )    
    
    print(f"   ✅ Best RF Parameters: Depth={best_rf_model.getOrDefault('maxDepth')}, Trees={best_rf_model.getOrDefault('numTrees')}")
    print(f"   🏆 Random Forest AUC: {rf_auc_score:.4f}")

####Visualize Predictions & Feature Importance
Let's show the business stakeholders why the Random Forest made its decisions by displaying Feature Importance.

In [0]:
import pandas as pd

# 1. Display sample predictions vs actuals
print("📊 Sample Predictions (Random Forest):")
display(
    rf_predictions.select("user_id", "purchased", "probability", "prediction")
    .filter("purchased = 1") # Show some actual purchasers to see the probabilities
    .limit(10)
)

# 2. Extract and Visualize Feature Importance
# This tells us which behavioral metric (Views, Carts, Events) is the strongest predictor of a purchase.
importances = best_rf_model.featureImportances.toArray()
feature_importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

spark_fi_df = spark.createDataFrame(feature_importance_df)

print("\n🧠 Feature Importance Breakdown:")
display(spark_fi_df)

# 💡 UI VISUALIZATION TIP:
# Click '+' -> 'Visualization' -> 'Bar Chart'.
# X-Axis: 'Feature', Y-Axis: 'Importance'.

Databricks visualization. Run in Databricks to view.